# QLoRA Fine-Tuning in Google Colab

This notebook fine-tunes Mistral-7B-Instruct with your capstone dataset using QLoRA.

It will:
1. Verify GPU availability
2. Install required packages
3. Upload `training_data.jsonl` and `fine_tune.py`
4. Run training
5. Zip and download the adapter output

In [ ]:
import subprocess
import torch

print('[+] GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('[+] GPU name:', torch.cuda.get_device_name(0))
    print('[+] CUDA version (torch):', torch.version.cuda)
    print('[+] Device count:', torch.cuda.device_count())
else:
    print('[!] No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU')

print('\n[+] nvidia-smi:')
try:
    result = subprocess.run(['nvidia-smi'], check=False, text=True, capture_output=True)
    print(result.stdout if result.stdout else result.stderr)
except Exception as e:
    print('[!] Could not run nvidia-smi:', e)

In [ ]:
from pathlib import Path

req_path = Path('requirements_finetune.txt')
if not req_path.exists():
    req_path.write_text(
        '\n'.join([
            'torch>=2.1.0',
            'transformers>=4.40.0',
            'peft>=0.10.0',
            'trl>=0.8.0',
            'bitsandbytes>=0.43.0',
            'datasets>=2.18.0',
            'accelerate>=0.28.0',
        ]) + '\n',
        encoding='utf-8'
    )

!pip install -q -r requirements_finetune.txt
print('[✓] Installed packages from requirements_finetune.txt')

In [ ]:
from google.colab import files
from pathlib import Path

print('[+] Upload training_data.jsonl and fine_tune.py')
uploaded = files.upload()

required_files = ['training_data.jsonl', 'fine_tune.py']
missing = [name for name in required_files if not Path(name).exists()]
if missing:
    raise FileNotFoundError(f'Missing required uploaded files: {missing}')

for name in required_files:
    print(f'[✓] Found {name}')

line_count = sum(1 for _ in open('training_data.jsonl', 'r', encoding='utf-8'))
print(f'[+] training_data.jsonl line count: {line_count}')

In [ ]:
!python fine_tune.py --data training_data.jsonl --out ./output

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('finetuned_adapters', 'zip', './output')
files.download('finetuned_adapters.zip')